# Jhadina Google Homebase — iPhone-friendly synthetic DVC Drive test

**What this proves:** a temporary Google Colab CPU session can use a *mounted* personal My Drive folder as DVC object storage, upload 64 random bytes, delete only the local scratch cache, and restore the bytes with matching SHA-256. No private Jhadina data is read or uploaded.

**What it does NOT prove:** the dvc-gdrive OAuth plugin, background worker authorization, PostgreSQL/MinIO/NATS recovery, or a production server. Colab is temporary, quotas and access may limit execution.

Run the two code cells individually from your Google account. Read the Drive mounting permissions before authorizing: Colab may receive broad access to your Drive. Only run a notebook you trust. It never deletes or shares Google Drive files.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')  # owner completes Google sign-in interactively


In [ ]:
import hashlib, json, os, secrets, shutil, subprocess, sys, tempfile
from pathlib import Path

# Expected folder names from the connected, personal My Drive.
allowed = Path('/content/drive/MyDrive/JHADINA-HOMEBASE/07-DVC-VERSIONED-ASSETS')
if not allowed.is_dir() or allowed.is_symlink():
    raise RuntimeError('Private DVC folder missing: verify the owner account and folder path')
# Write only to this dedicated harmless-test subfolder. Never use My Drive root.
remote = allowed / '_JHADINA_COLAB_SYNTHETIC_DVC'
if remote.exists() and (remote.is_symlink() or not remote.is_dir()):
    raise RuntimeError('Unsafe existing synthetic DVC test destination')

subprocess.run([sys.executable, '-m', 'pip', 'install', '--quiet', 'dvc'], check=True, timeout=240)
with tempfile.TemporaryDirectory(prefix='jhadina-colab-dvc-', dir='/content') as scratch:
    root = Path(scratch)
    os.chmod(root, 0o700)
    def dvc(*args):
        subprocess.run(['dvc', *args], check=True, cwd=root, timeout=300)
    dvc('init', '--no-scm')
    dvc('remote', 'add', 'test_drive_mount', str(remote))
    data = secrets.token_bytes(64)
    expected = hashlib.sha256(data).hexdigest()
    obj = root / 'synthetic-canary.bin'
    obj.write_bytes(data)
    dvc('add', obj.name)
    dvc('push', '-r', 'test_drive_mount', obj.name + '.dvc')
    obj.unlink()  # local synthetic test file only
    cache = root / '.dvc' / 'cache'
    if not cache.is_dir() or cache.is_symlink() or cache.resolve() != (root / '.dvc' / 'cache').resolve():
        raise RuntimeError('Scratch DVC cache not verified; cannot prove remote-only restore')
    shutil.rmtree(cache)  # deletes only /content temporary local cache; never Google Drive
    dvc('pull', '-r', 'test_drive_mount', obj.name + '.dvc')
    if not obj.is_file() or hashlib.sha256(obj.read_bytes()).hexdigest() != expected:
        raise RuntimeError('DVC remote-only restore hash verification FAILED')
    print(json.dumps({'schema':'jhadina.google-homebase.colab-mounted-dvc-canary.v1',
                      'synthetic_bytes':64, 'sha256':expected,
                      'mounted_personal_drive_remote_roundtrip_verified':True,
                      'dvc_gdrive_plugin_authenticated':False,
                      'machine_google_oauth_verified':False,
                      'production_backup_restored':False}, indent=2))
